# Julia baseline — original full model versus new modulated backend

This uses the **existing** modWorm Julia PPC neurons/body and CVODE_BDF/Tsit5, following *1.4 Custom start (Full model)*. We first compare with zero neuromodulation and identical initial states. The notebook does **not** claim numerical agreement before measuring it. Julia compilation can make the first run slow.

In [ ]:
from pathlib import Path
import os, time
import numpy as np
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'modWorm'/'__init__.py').exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within your modWorm checkout")
os.chdir(ROOT)

from modWorm import predefined_classes_nv as nv, utils
from modWorm import predefined_classes_mb as mb
from modWorm import proprioception_simulation as sim
from modWorm import animation
from modWorm.julia_multiscale import run_modulated_julia, run_independent_julia_population, JuliaRunSettings
from modWorm.serotonin.serotonin_modulation import SerotoninModulation
from modWorm.pdf_modulation import PDFR1Modulation

GAP, SYN = utils.construct_connectome_Cook()
MUSCLE_MAP = np.load('modWorm/muscle_maps/muscle_map_adjust.npy')
PRESET = np.load('modWorm/presets_input/input_mat_gentle_post_touch.npy')
print('Loaded canonical', GAP.shape, SYN.shape, MUSCLE_MAP.shape, PRESET.shape)

def make_worm(seed=17, *, x=0.0, y=0.0, angle=0.0):
    # Both model constructors consume NumPy global RNG (legacy contract).
    np.random.seed(seed)
    neural = nv.CelegansWorm_NervousSystem_PPC_Julia(GAP, SYN)
    body = mb.CelegansWorm_MuscleBody_PPC_Julia(MUSCLE_MAP)
    body.initcond = np.asarray(body.initcond, dtype=float).copy()
    body.initcond[0] += x
    body.initcond[1] += y
    body.initcond[2:2+body.body_SegCount] += angle
    return neural, body

# Each sample represents 0.01 s in the original PPC class.
STEPS = 4  # 0.03 simulated seconds. Increase only after profiling.
INPUTS = PRESET[:STEPS].copy()
SETTINGS = JuliaRunSettings()


In [ ]:
import traceback
from pathlib import Path
import time

neural_a, body_a = make_worm(seed=10)

try:
    start = time.perf_counter()
    legacy = sim.run_network_julia(neural_a, body_a, INPUTS)
    print("Legacy Julia succeeded in", time.perf_counter() - start, "seconds")

except Exception as exc:
    full_error = "".join(
        traceback.format_exception(type(exc), exc, exc.__traceback__)
    )

    output = Path("julia_full_error.txt")
    output.write_text(full_error, encoding="utf-8")

    print("Exception type:", type(exc).__name__)
    print("\n--- JULIA ERROR (FIRST 6,000 CHARACTERS) ---\n")
    print(str(exc)[:6000])
    print("\nComplete traceback saved to:", output.resolve())

In [ ]:
import inspect
from pathlib import Path

source = inspect.getsource(sim.run_network_julia)
Path("julia_runner_source.txt").write_text(source, encoding="utf-8")

print("Runner:", sim.__file__)
print("Source saved to julia_runner_source.txt")

In [ ]:
neural_a, body_a = make_worm(seed=10)
neural_b, body_b = make_worm(seed=10)
np.testing.assert_array_equal(neural_a.initcond, neural_b.initcond)
np.testing.assert_array_equal(body_a.initcond, body_b.initcond)

start=time.perf_counter()
legacy = sim.run_network_julia(neural_a, body_a, INPUTS)
t_legacy=time.perf_counter()-start
start=time.perf_counter()
new = run_modulated_julia(neural_b, body_b, INPUTS, settings=SETTINGS)
t_new=time.perf_counter()-start
for variable in ('v_solution','s_solution','muscle_force','x_solution','y_solution'):
    a=np.asarray(legacy[variable],dtype=float)
    b=np.asarray(new[variable],dtype=float)
    print(variable, a.shape, b.shape, 'max absolute difference', np.max(np.abs(a-b)))
print('Runtime including initial compilation (not speed benchmark):',t_legacy,t_new)


In [ ]:
plt.figure(figsize=(7,5))
plt.plot(np.asarray(legacy['x_solution'])[:,0],np.asarray(legacy['y_solution'])[:,0],label='original Julia')
plt.plot(new['x_solution'][:,0],new['y_solution'][:,0], '--',label='new zero-mod Julia')
plt.legend(); plt.xlabel('Body x (model units)'); plt.ylabel('Body y (model units)'); plt.title('Head trajectory — short numerical check'); plt.show()

## Next step

After comparing errors above, increase STEPS gradually. Never infer biological roaming/dwelling from a few hundredths of a second. Keep the original neural/body parameters (including fluid viscosity) identical in both arms.

In [ ]:
# =============================================================
# MODWORM DIAGNOSTIC  —  run top to bottom, paste the whole output back
# =============================================================
import sys, os, time, traceback, json
import numpy as np
from pathlib import Path

SEP = "=" * 78

def hdr(t): print(f"\n{SEP}\n{t}\n{SEP}")
def sub(t): print(f"\n--- {t} ---")

# ---------- 0. environment ----------
hdr("0. ENVIRONMENT")
print("python       :", sys.version.split()[0])
print("numpy        :", np.__version__)
print("cwd          :", os.getcwd())
try:
    from juliacall import Main as jl
    print("juliacall    :", jl.seval('string(VERSION)'))
    print("julia bindir :", jl.seval('Sys.BINDIR'))
    print("active proj  :", jl.seval('Base.active_project()'))
    try:
        pkgs = jl.seval('join(sort([string(k," ",v) for (k,v) in Pkg.dependencies()]), "\\n")')
        print("julia pkgs   :\n", pkgs[:2000])
    except Exception as e:
        print("julia pkgs   : (could not enumerate:", e, ")")
except Exception as e:
    print("juliacall    : FAILED:", repr(e))

# ---------- 1. static data finiteness ----------
hdr("1. STATIC DATA")
for name, arr in {"GAP": GAP, "SYN": SYN, "MUSCLE_MAP": MUSCLE_MAP,
                  "PRESET": PRESET, "INPUTS": INPUTS}.items():
    a = np.asarray(arr, dtype=float)
    print(f"{name:12s} shape={str(a.shape):15s} finite={np.isfinite(a).all()} "
          f"min={np.nanmin(a):+.4g} max={np.nanmax(a):+.4g} "
          f"n_nan={np.isnan(a).sum()} n_inf={np.isinf(a).sum()}")

# ---------- 2. build worm, dump python-side attributes ----------
hdr("2. WORM CONSTRUCTION (seed=10, like the failing cell)")
try:
    neural_a, body_a = make_worm(seed=10)
    print("neural type  :", type(neural_a).__name__)
    print("body   type  :", type(body_a).__name__)
    print("neural initcond shape:", np.shape(neural_a.initcond),
          "finite:", np.isfinite(np.asarray(neural_a.initcond, float)).all())
    print("body   initcond shape:", np.shape(body_a.initcond),
          "finite:", np.isfinite(np.asarray(body_a.initcond, float)).all())
    # Common body attrs; print whichever exist
    for attr in ["body_SegCount", "body_Mass", "body_SegLength", "body_RadiusV",
                 "body_RadiusH", "body_Area", "body_Stiff", "body_StiffSeg",
                 "body_StiffMotion", "body_InertiaSeg", "body_InertiaMotion",
                 "body_ForceTransform", "muscle_Map", "muscle_MapInv",
                 "fluid_Viscosity", "fluid_Density", "timescale", "maxiters"]:
        if hasattr(body_a, attr):
            v = np.asarray(getattr(body_a, attr), dtype=float)
            finite = np.isfinite(v).all() if v.size else True
            flag = "" if finite else "   <-- NON-FINITE"
            mn = np.nanmin(v) if v.size else float("nan")
            mx = np.nanmax(v) if v.size else float("nan")
            print(f"  body.{attr:22s} shape={str(v.shape):15s} finite={finite} "
                  f"min={mn:+.4g} max={mx:+.4g}{flag}")
    for attr in ["network_Size", "timescale", "tf", "nsteps"]:
        if hasattr(neural_a, attr):
            v = getattr(neural_a, attr)
            print(f"  neural.{attr:20s} = {v}")
except Exception:
    print("BUILD FAILED:")
    traceback.print_exc()

# ---------- 3. Julia-side parameter dict ----------
hdr("3. JULIA PARAM DICT (modworm_native_ppc_dict)")
try:
    from juliacall import Main as jl
    # mimic run_network_julia: it builds vars_NN_mb from the body object.
    # Try several common constructors; whichever works, we use.
    d_mb = None
    for expr in [
        "Main.modworm_native_ppc_dict(pyconvert_attr)",
    ]:
        pass  # placeholder, we do it below

    # The original runner converts the object's __dict__; try that.
    body_vars = dict(body_a.__dict__) if hasattr(body_a, "__dict__") else None
    if body_vars is not None:
        # Add the fields the runner sets (maxiters, id) if missing.
        body_vars.setdefault("maxiters", getattr(body_a, "maxiters", 1e6))
        body_vars.setdefault("id", 0)
        try:
            d_mb = jl.modworm_native_ppc_dict(body_vars)
            # Convert Julia dict-like to python dict
            keys = list(jl.keys(d_mb))
            print("julia returned", len(keys), "keys")
            for k in keys:
                v = jl.getindex(d_mb, k)
                try:
                    a = np.asarray(jl.Array(v) if hasattr(jl, "Array") else v, dtype=float)
                except Exception:
                    try:
                        a = np.asarray(list(v), dtype=float)
                    except Exception:
                        a = None
                if a is None:
                    print(f"  {str(k):24s} (non-array) {type(v)}")
                    continue
                finite = np.isfinite(a).all() if a.size else True
                flag = "" if finite else "   <-- NON-FINITE"
                mn = np.nanmin(a) if a.size else float("nan")
                mx = np.nanmax(a) if a.size else float("nan")
                print(f"  {str(k):24s} shape={str(a.shape):15s} finite={finite} "
                      f"min={mn:+.4g} max={mx:+.4g}{flag}")
        except Exception:
            print("julia param-dict build FAILED:")
            traceback.print_exc()
    else:
        print("body object has no __dict__; cannot introspect")
except Exception:
    traceback.print_exc()

# ---------- 4. mass-matrix probe at t=0 ----------
hdr("4. MASS MATRIX PROBE AT t=0")
try:
    from juliacall import Main as jl
    # Build the same M the RHS builds, using the julia-side parameters.
    # We locate fwd_body_D2Angles / fwd_body_mass_matrix dynamically.
    probe_src = r'''
    function __diag_probe_mass(p, u)
        n = Int(p.body_SegCount)
        phi    = u[3:2+n]
        phidot = u[3+n:2+2n]
        # Try a named mass-matrix helper first.
        for fname in (:fwd_body_mass_matrix, :fwd_body_M, :body_mass_matrix)
            if isdefined(Main, fname)
                M = Base.invokelatest(getfield(Main, fname), p, phi, phidot)
                return (name=string(fname), n=n, size=size(M),
                        n_nan=sum(isnan, M), n_inf=sum(isinf, M),
                        cond=try cond(M) catch; NaN end,
                        first_bad=findall(x->!isfinite(x), vec(M))[1:min(end,10)])
            end
        end
        # Fallback: replicate the call inside fwd_body_D2Angles but with
        # zeroed coupling terms, to isolate the geometric mass matrix.
        Mdiff   = zeros(n); Wy_diff = zeros(n); Wx_diff = zeros(n)
        # Call fwd_body_D2Angles directly and catch anything thrown
        try
            out = Base.invokelatest(getfield(Main, :fwd_body_D2Angles),
                                    p, phi, phidot, Mdiff, Wy_diff, Wx_diff)
            return (name="fwd_body_D2Angles(zeros)", n=n,
                    size=(length(out),), n_nan=sum(isnan, out),
                    n_inf=sum(isinf, out), cond=NaN, first_bad=Int[])
        catch err
            return (name="fwd_body_D2Angles(zeros)", error=string(err))
        end
    end
    '''
    jl.seval(probe_src)
    # Build a minimal p dict for the probe
    if d_mb is not None:
        u0 = np.asarray(body_a.initcond, dtype=float)
        res = jl.seval("__diag_probe_mass")(d_mb, jl.Vector(u0))
        print("probe result:", res)
    else:
        print("no d_mb; skipping mass-matrix probe")
except Exception:
    print("mass-matrix probe FAILED:")
    traceback.print_exc()

# ---------- 5. minimal STEP=1 run (isolate first-step blow-up) ----------
hdr("5. MINIMAL RUN: STEPS = 1")
try:
    neural_b, body_b = make_worm(seed=10)
    inputs1 = np.asarray(PRESET[:1], dtype=float)
    t0 = time.perf_counter()
    out1 = sim.run_network_julia(neural_b, body_b, inputs1)
    dt = time.perf_counter() - t0
    print(f"STEPS=1 run OK in {dt:.2f}s")
    for k in ("v_solution", "s_solution", "muscle_force", "x_solution", "y_solution"):
        if k in out1:
            a = np.asarray(out1[k], dtype=float)
            print(f"  {k:14s} shape={str(a.shape):15s} "
                  f"finite={np.isfinite(a).all()} "
                  f"nan={np.isnan(a).sum()} inf={np.isinf(a).sum()}")
except Exception as e:
    print("STEPS=1 FAILED:", type(e).__name__)
    print(str(e)[:4000])

# ---------- 6. try steps 1,2,3,4 to find first divergence ----------
hdr("6. STEP SWEEP: locate the first failing step count")
for n in (1, 2, 3, 4):
    try:
        nb, bb = make_worm(seed=10)
        out = sim.run_network_julia(nb, bb, np.asarray(PRESET[:n], dtype=float))
        mx = max(
            np.nanmax(np.abs(np.asarray(out[k], float)))
            for k in ("v_solution", "s_solution", "muscle_force",
                      "x_solution", "y_solution") if k in out
        )
        print(f"  STEPS={n}: OK   max |value| = {mx:.4g}")
    except Exception as e:
        print(f"  STEPS={n}: FAIL {type(e).__name__}: {str(e).splitlines()[0][:200]}")

# ---------- 7. small facts to correlate ----------
hdr("7. SIZE SANITY")
print("neural initcond len:", len(np.asarray(neural_a.initcond)))
print("body   initcond len:", len(np.asarray(body_a.initcond)))
print("MUSCLE_MAP rows    :", MUSCLE_MAP.shape[0])
print("GAP  shape         :", GAP.shape)
print("SYN  shape         :", SYN.shape)
print("INPUTS shape       :", np.shape(INPUTS))
for attr in ("body_SegCount", "network_Size", "nsteps", "tf", "timescale"):
    for obj, label in ((neural_a, "neural"), (body_a, "body")):
        if hasattr(obj, attr):
            print(f"{label}.{attr:16s} = {getattr(obj, attr)}")

print("\nDIAGNOSTIC COMPLETE.")

In [ ]:
# =============================================================
# FOLLOW-UP DIAGNOSTIC A  —  body stability, zero input
# =============================================================
import numpy as np, traceback
SEP = "=" * 78

print(SEP); print("A1. ZERO-INPUT BODY RUN"); print(SEP)
try:
    nb, bb = make_worm(seed=10)
    zero_inputs = np.zeros((4, 279), dtype=float)   # NO external drive
    out = sim.run_network_julia(nb, bb, zero_inputs)
    for k in ("v_solution","s_solution","muscle_force","x_solution","y_solution"):
        if k in out:
            a = np.asarray(out[k], float)
            print(f"  {k:14s} shape={str(a.shape):16s} "
                  f"finite={np.isfinite(a).all()} "
                  f"nan={np.isnan(a).sum()} inf={np.isinf(a).sum()} "
                  f"max|.|={np.nanmax(np.abs(a)):.4g}")
except Exception as e:
    print("ZERO-INPUT FAILED:", type(e).__name__)
    print(str(e)[:3000])

print()
print(SEP); print("A2. TINY-INPUT BODY RUN (input * 1e-3)"); print(SEP)
try:
    nb, bb = make_worm(seed=10)
    tiny = np.asarray(PRESET[:4], float) * 1e-3
    out = sim.run_network_julia(nb, bb, tiny)
    for k in ("v_solution","s_solution","muscle_force","x_solution","y_solution"):
        if k in out:
            a = np.asarray(out[k], float)
            print(f"  {k:14s} shape={str(a.shape):16s} "
                  f"finite={np.isfinite(a).all()} "
                  f"max|.|={np.nanmax(np.abs(a)):.4g}")
except Exception as e:
    print("TINY-INPUT FAILED:", type(e).__name__)
    print(str(e)[:3000])

print()
print(SEP); print("A3. TALLER INPUT (STEPS=10, small amplitude)"); print(SEP)
try:
    nb, bb = make_worm(seed=10)
    small = np.asarray(PRESET[:10], float) * 1e-2
    out = sim.run_network_julia(nb, bb, small)
    print("  10-step small-amplitude run OK")
    for k in ("v_solution","s_solution","muscle_force","x_solution","y_solution"):
        if k in out:
            a = np.asarray(out[k], float)
            print(f"  {k:14s} finite={np.isfinite(a).all()} max|.|={np.nanmax(np.abs(a)):.4g}")
except Exception as e:
    print("SMALL-AMPLITUDE FAILED:", type(e).__name__)
    print(str(e)[:3000])

print()
print(SEP); print("A4. BODY GEOMETRY SANITY"); print(SEP)
for attr in ("body_SegCount","body_SegLength","body_Mass","body_InertiaMotion",
             "body_InertiaSeg","body_Stiff","body_StiffSeg","body_StiffMotion",
             "body_ForceTransform","body_ForceScaling","fluid_Viscosity",
             "fluid_Density","timescale","body_DragCoeff","body_CurvatureScaling",
             "body_CurvatureForcing","body_E","body_CalciumFactor"):
    if hasattr(body_a, attr):
        v = getattr(body_a, attr)
        a = np.asarray(v, float) if not isinstance(v, (int, float, bool)) else np.array([v])
        print(f"  body.{attr:24s} shape={str(a.shape):16s} "
              f"finite={np.isfinite(a).all()} "
              f"min={np.nanmin(a):+.4g} max={np.nanmax(a):+.4g}")

print()
print(SEP); print("A5. INPUT SCALE SANITY"); print(SEP)
print(f"  PRESET shape={PRESET.shape} dtype={PRESET.dtype}")
print(f"  PRESET first row  : min={PRESET[0].min():.4g} max={PRESET[0].max():.4g}")
print(f"  PRESET first col  : min={PRESET[:,0].min():.4g} max={PRESET[:,0].max():.4g}")
print(f"  any negative      : {(PRESET < 0).any()}")
print(f"  any non-finite    : {(~np.isfinite(PRESET)).any()}")

print("\nFOLLOW-UP A COMPLETE.")